<a href="https://colab.research.google.com/github/anuragjaine/credit-risk-modelling/blob/main/notebooks/04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Feature Engineering & Model Preparation

## Objectives
1. Create new domain-driven features
2. Encode categorical variables for ML models
3. Perform TIME-BASED train-test split (not random)
4. Scale numerical features
5. Save prepared datasets for modelling

## Why Time-Based Split?
In real-world deployment, we train models on past loans and
predict future loans. A random split would leak future information
into training, giving us misleading performance estimates.

We will use:
- Train: loans issued before 2017
- Test: loans issued in 2017 and later

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_style('whitegrid')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete")

Mounted at /content/drive
Setup complete


In [2]:
# Load cleaned dataset
file_path = "/content/drive/MyDrive/credit-risk-modelling/data/processed/loans_cleaned.csv"
df = pd.read_csv(file_path)

# Convert dates
df['issue_d'] = pd.to_datetime(df['issue_d'])

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean() * 100:.2f}%")
df.head()

Shape: (1345310, 30)
Default rate: 19.96%


,target,loan_status,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,issue_d,emp_length,home_ownership,annual_inc,verification_status,dti,addr_state,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,delinq_2yrs,inq_last_6mths,mort_acc,pub_rec_bankruptcies,application_type,initial_list_status,credit_history_years,fico_avg
0,0,Fully Paid,3600.0,36,13.99,123.03,C,C4,debt_consolidation,2015-12-01,10,MORTGAGE,55000.0,Not Verified,5.91,PA,2003-08-01,7.0,0.0,2765.0,29.7,13.0,0.0,1.0,1.0,0.0,Individual,w,12.342466,677.0
1,0,Fully Paid,24700.0,36,11.99,820.28,C,C1,small_business,2015-12-01,10,MORTGAGE,65000.0,Not Verified,16.06,SD,1999-12-01,22.0,0.0,21470.0,19.2,38.0,1.0,4.0,4.0,0.0,Individual,w,16.010959,717.0
2,0,Fully Paid,20000.0,60,10.78,432.66,B,B4,home_improvement,2015-12-01,10,MORTGAGE,63000.0,Not Verified,10.78,IL,2000-08-01,6.0,0.0,7869.0,56.2,18.0,0.0,0.0,5.0,0.0,Joint App,w,15.342466,697.0
3,0,Fully Paid,10400.0,60,22.45,289.91,F,F1,major_purchase,2015-12-01,3,MORTGAGE,104433.0,Source Verified,25.37,PA,1998-06-01,12.0,0.0,21929.0,64.5,35.0,1.0,3.0,6.0,0.0,Individual,w,17.512329,697.0
4,0,Fully Paid,11950.0,36,13.44,405.18,C,C3,debt_consolidation,2015-12-01,4,RENT,34000.0,Source Verified,10.20,GA,1987-10-01,5.0,0.0,8822.0,68.4,6.0,0.0,0.0,0.0,0.0,Individual,w,28.186301,692.0


In [3]:
# Feature 1: Loan amount to income ratio
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

# Feature 2: Installment to monthly income ratio
df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12)

# Feature 3: High utilization flag
df['high_util_flag'] = (df['revol_util'] > 80).astype(int)

# Feature 4: Has public record flag
df['has_pub_rec'] = (df['pub_rec'] > 0).astype(int)

# Feature 5: Has delinquency flag
df['has_delinq'] = (df['delinq_2yrs'] > 0).astype(int)

# Feature 6: Has bankruptcy flag
df['has_bankruptcy'] = (df['pub_rec_bankruptcies'] > 0).astype(int)

# Feature 7: Multiple recent inquiries flag
df['multiple_inq_flag'] = (df['inq_last_6mths'] >= 3).astype(int)

# Feature 8: Credit history bucket
df['short_credit_history'] = (df['credit_history_years'] < 5).astype(int)

# Feature 9: Income bracket
df['low_income_flag'] = (df['annual_inc'] < 40000).astype(int)

# Feature 10: High DTI flag
df['high_dti_flag'] = (df['dti'] > 30).astype(int)

print("New features created:")
new_features = ['loan_to_income', 'installment_to_income', 'high_util_flag',
                'has_pub_rec', 'has_delinq', 'has_bankruptcy',
                'multiple_inq_flag', 'short_credit_history',
                'low_income_flag', 'high_dti_flag']

for feat in new_features:
    print(f"  ✓ {feat}")

print(f"\nNew shape: {df.shape}")

New features created:
  ✓ loan_to_income
  ✓ installment_to_income
  ✓ high_util_flag
  ✓ has_pub_rec
  ✓ has_delinq
  ✓ has_bankruptcy
  ✓ multiple_inq_flag
  ✓ short_credit_history
  ✓ low_income_flag
  ✓ high_dti_flag

New shape: (1345310, 40)


In [4]:
# Check default rates for new flag features
print("Default rate by new flags:\n")

for flag in ['high_util_flag', 'has_pub_rec', 'has_delinq',
             'has_bankruptcy', 'multiple_inq_flag', 'short_credit_history',
             'low_income_flag', 'high_dti_flag']:
    rate_0 = df[df[flag] == 0]['target'].mean() * 100
    rate_1 = df[df[flag] == 1]['target'].mean() * 100
    diff = rate_1 - rate_0
    print(f"{flag:30s} | Flag=0: {rate_0:5.2f}% | Flag=1: {rate_1:5.2f}% | Diff: +{diff:.2f}%")

Default rate by new flags:

high_util_flag                 | Flag=0: 19.49% | Flag=1: 22.72% | Diff: +3.23%
has_pub_rec                    | Flag=0: 19.39% | Flag=1: 22.75% | Diff: +3.36%
has_delinq                     | Flag=0: 19.61% | Flag=1: 21.45% | Diff: +1.84%
has_bankruptcy                 | Flag=0: 19.58% | Flag=1: 22.65% | Diff: +3.07%
multiple_inq_flag              | Flag=0: 19.58% | Flag=1: 26.74% | Diff: +7.15%
short_credit_history           | Flag=0: 19.85% | Flag=1: 24.20% | Diff: +4.35%
low_income_flag                | Flag=0: 19.26% | Flag=1: 23.75% | Diff: +4.48%
high_dti_flag                  | Flag=0: 18.99% | Flag=1: 29.17% | Diff: +10.18%


### 🔍 Insight
All our flag features show meaningfully higher default rates when activated.
This confirms these are good predictors for the model.

In [5]:
# Drop EDA binning columns if they exist
cols_to_drop = ['int_rate_bin', 'income_bin', 'fico_bin', 'dti_bin', 'issue_year']
cols_to_drop = [col for col in cols_to_drop if col in df.columns]

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped: {cols_to_drop}")
else:
    print("No EDA columns to drop")

# Also drop loan_status (we have target already)
if 'loan_status' in df.columns:
    df = df.drop(columns=['loan_status'])
    print("Dropped: loan_status")

print(f"Shape: {df.shape}")

No EDA columns to drop
Dropped: loan_status
Shape: (1345310, 39)


In [6]:
# Check categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['grade', 'sub_grade', 'purpose', 'home_ownership', 'verification_status', 'addr_state', 'earliest_cr_line', 'application_type', 'initial_list_status']


In [7]:
# Strategy:
# - grade, sub_grade: ordinal encoding (A=1, B=2, ..., G=7)
# - purpose, home_ownership, etc: one-hot encoding

# Ordinal encoding for grade
grade_mapping = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_mapping)

# Ordinal encoding for sub_grade (A1=1, A2=2, ..., G5=35)
sub_grade_list = [f"{g}{i}" for g in 'ABCDEFG' for i in range(1, 6)]
sub_grade_mapping = {sg: i+1 for i, sg in enumerate(sub_grade_list)}
df['sub_grade'] = df['sub_grade'].map(sub_grade_mapping)

print("Grade encoding complete")
print(f"Grade unique values: {sorted(df['grade'].unique())}")
print(f"Sub_grade range: {df['sub_grade'].min()} to {df['sub_grade'].max()}")

Grade encoding complete
Grade unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Sub_grade range: 1 to 35


In [8]:
# One-hot encoding for nominal categories
nominal_cols = ['home_ownership', 'verification_status', 'purpose',
                'application_type', 'initial_list_status', 'addr_state']

# Keep only those that exist
nominal_cols = [col for col in nominal_cols if col in df.columns]

# Apply one-hot encoding
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

print(f"After one-hot encoding shape: {df.shape}")
print(f"\nNew columns count: {df.shape[1]}")

After one-hot encoding shape: (1345310, 105)

New columns count: 105


In [10]:
# Drop earliest_cr_line - we already extracted credit_history_years from it
if 'earliest_cr_line' in df.columns:
    df = df.drop(columns=['earliest_cr_line'])
    print("Dropped: earliest_cr_line")

# Verify all features are now numerical (except issue_d which we need for splitting)
object_cols = df.select_dtypes(include=['object']).columns.tolist()
if object_cols:
    print(f"Warning: Still object columns: {object_cols}")
else:
    print("✅ All features are numerical (except issue_d which we keep for splitting)")

print(f"\nFinal shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes.value_counts())

Dropped: earliest_cr_line
✅ All features are numerical (except issue_d which we keep for splitting)

Final shape: (1345310, 104)

Data types:
int64             85
float64           18
datetime64[ns]     1
Name: count, dtype: int64


In [11]:
# Sort by issue date
df = df.sort_values('issue_d').reset_index(drop=True)

# Check date distribution
print("Loan distribution by year:")
print(df['issue_d'].dt.year.value_counts().sort_index())

Loan distribution by year:
issue_d
2007       251
2008      1562
2009      4716
2010     11536
2011     21721
2012     53367
2013    134804
2014    223102
2015    375545
2016    293095
2017    169300
2018     56311
Name: count, dtype: int64


In [12]:
# Use 2017 as cutoff
split_date = '2017-01-01'

train_df = df[df['issue_d'] < split_date].copy()
test_df = df[df['issue_d'] >= split_date].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Train default rate: {train_df['target'].mean()*100:.2f}%")
print(f"Test default rate:  {test_df['target'].mean()*100:.2f}%")
print(f"\nTrain date range: {train_df['issue_d'].min()} to {train_df['issue_d'].max()}")
print(f"Test date range:  {test_df['issue_d'].min()} to {test_df['issue_d'].max()}")

Train shape: (1119699, 104)
Test shape:  (225611, 104)
Train default rate: 19.70%
Test default rate:  21.28%

Train date range: 2007-06-01 00:00:00 to 2016-12-01 00:00:00
Test date range:  2017-01-01 00:00:00 to 2018-12-01 00:00:00


In [13]:
# Drop date column (we won't use it as feature)
train_df = train_df.drop(columns=['issue_d'])
test_df = test_df.drop(columns=['issue_d'])

# Separate features (X) and target (y)
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']

X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (1119699, 102)
y_train shape: (1119699,)
X_test shape:  (225611, 102)
y_test shape:  (225611,)


In [14]:
# Final check for missing values
print("Missing in X_train:", X_train.isnull().sum().sum())
print("Missing in X_test: ", X_test.isnull().sum().sum())

# Fill any remaining with median
for col in X_train.columns:
    if X_train[col].isnull().sum() > 0:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        print(f"Filled {col} with {median_val}")

print(f"\nFinal check:")
print(f"X_train missing: {X_train.isnull().sum().sum()}")
print(f"X_test missing:  {X_test.isnull().sum().sum()}")

Missing in X_train: 0
Missing in X_test:  0

Final check:
X_train missing: 0
X_test missing:  0


In [22]:
# Check for infinity values
print("Checking for infinity values...")

# Replace infinity with NaN, then fill with median
for col in X_train.columns:
    if np.isinf(X_train[col]).any():
        # Count infinities
        inf_count = np.isinf(X_train[col]).sum()
        print(f"  {col}: {inf_count} infinity values found")

        # Replace infinity with NaN
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)

        # Fill with median from train
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

        print(f"    → Replaced with median: {median_val:.4f}")

# Final verification
print(f"\n✅ Infinity check complete")
print(f"X_train infinities: {np.isinf(X_train.values).sum()}")
print(f"X_test infinities:  {np.isinf(X_test.values).sum()}")
print(f"X_train NaN: {X_train.isnull().sum().sum()}")
print(f"X_test NaN:  {X_test.isnull().sum().sum()}")

Checking for infinity values...
  loan_to_income: 35 infinity values found
    → Replaced with median: 0.2000
  installment_to_income: 35 infinity values found
    → Replaced with median: 0.0733

✅ Infinity check complete
X_train infinities: 0
X_test infinities:  0
X_train NaN: 0
X_test NaN:  0


In [23]:
# Scale features using StandardScaler
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling complete")
print(f"\nX_train_scaled mean (should be ~0): {X_train_scaled.mean().mean():.4f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std().mean():.4f}")

Scaling complete

X_train_scaled mean (should be ~0): -0.0000
X_train_scaled std (should be ~1): 1.0000


In [24]:
import os

# Define output paths
output_dir = "/content/drive/MyDrive/credit-risk-modelling/data/processed"
os.makedirs(output_dir, exist_ok=True)

# Save unscaled versions (for tree models)
X_train.to_csv(f"{output_dir}/X_train.csv", index=False)
X_test.to_csv(f"{output_dir}/X_test.csv", index=False)

# Save scaled versions (for logistic regression)
X_train_scaled.to_csv(f"{output_dir}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{output_dir}/X_test_scaled.csv", index=False)

# Save targets
y_train.to_csv(f"{output_dir}/y_train.csv", index=False)
y_test.to_csv(f"{output_dir}/y_test.csv", index=False)

# Save scaler for future use
import joblib
joblib.dump(scaler, f"{output_dir}/scaler.pkl")

print("All files saved successfully:")
print(f"  - X_train.csv: {X_train.shape}")
print(f"  - X_test.csv: {X_test.shape}")
print(f"  - X_train_scaled.csv: {X_train_scaled.shape}")
print(f"  - X_test_scaled.csv: {X_test_scaled.shape}")
print(f"  - y_train.csv: {y_train.shape}")
print(f"  - y_test.csv: {y_test.shape}")
print(f"  - scaler.pkl")

All files saved successfully:
  - X_train.csv: (1119699, 102)
  - X_test.csv: (225611, 102)
  - X_train_scaled.csv: (1119699, 102)
  - X_test_scaled.csv: (225611, 102)
  - y_train.csv: (1119699,)
  - y_test.csv: (225611,)
  - scaler.pkl


## Notebook 04 Summary

### What We Did
1. ✅ Created 10 new engineered features based on credit risk intuition
2. ✅ Encoded loan grade ordinally (A=1, ..., G=7)
3. ✅ One-hot encoded purpose, home_ownership, state, etc.
4. ✅ Performed TIME-BASED train-test split (2017 cutoff)
5. ✅ Scaled features using StandardScaler
6. ✅ Saved both scaled and unscaled versions

### Key Design Decisions
- **Time-based split** prevents future information leakage
- **Two versions** of data (scaled/unscaled) for different model types
- **Scaler trained on train only** to prevent data leakage
- **Ordinal encoding for grade** preserves natural ordering

### Dataset Sizes
- Train: ~1.1M loans (pre-2017)
- Test: ~200K loans (2017-2018)

### Next Steps (Notebook 05)
- Build baseline Logistic Regression model
- Build Random Forest model
- Build XGBoost model
- Compare using AUC-ROC, KS Statistic, Precision-Recall